# Entrega 1: Proyecto Soluciones Intensivas en Datos (2026 -02)

# Notebook 1: Ingesta de datos

## OceanWatch Analytics

### Presentado por: Gupo 2

#### Pablo Daniel Díaz Mera
#### Ana Catalina Gelvez
#### Christian Rene Mendez

## 0. Creación del espacio de trabajo

In [0]:
CATALOGO = "workspace"
ESQUEMA = "oceanwatch"
VOLUMEN = "ais_data"

# Crear esquema
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{ESQUEMA}
""")

# Crear Volume
spark.sql(f"""
CREATE VOLUME IF NOT EXISTS {CATALOGO}.{ESQUEMA}.{VOLUMEN}
""")

DataFrame[]

Rutas de almacenamiento

In [0]:
BASE = f"/Volumes/{CATALOGO}/{ESQUEMA}/{VOLUMEN}"

RUTA_ZIP = f"{BASE}/zip"
RUTA_CSV = f"{BASE}/csv"

print("Ruta base:", BASE)
print("ZIP:", RUTA_ZIP)
print("CSV:", RUTA_CSV)

Ruta base: /Volumes/workspace/oceanwatch/ais_data
ZIP: /Volumes/workspace/oceanwatch/ais_data/zip
CSV: /Volumes/workspace/oceanwatch/ais_data/csv


Creación de las carpetas

In [0]:

dbutils.fs.mkdirs(RUTA_ZIP)
dbutils.fs.mkdirs(RUTA_CSV)

True

## 1. Ingesta de los datos

Definición de las rutas de descarga de los datos

In [0]:
BASE_URL = "https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023"

archivos = [
    f"AIS_2023_06_{dia:02d}.zip"
    for dia in range(1, 8)
]

urls = [
    f"{BASE_URL}/{archivo}"
    for archivo in archivos
]

for url in urls:
    print(url)

https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_01.zip
https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_02.zip
https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_03.zip
https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_04.zip
https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_05.zip
https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_06.zip
https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_07.zip


Función para la descarga de los archivos

In [0]:

import os
import time
import requests

MAX_REINTENTOS = 3
CHUNK_SIZE = 1024 * 1024  # 1 MB


def descargar_archivo(url, ruta_destino):
    
    # Si ya existe, no volver a descargarlo
    if os.path.exists(ruta_destino):
        print(f"Ya existe: {os.path.basename(ruta_destino)}")
        return

    #Se realiza el reintento de descarga un máximo de MAX_REINTENTOS veces
    for intento in range(1, MAX_REINTENTOS + 1):
        try:
            print(
                f"Descargando {os.path.basename(ruta_destino)} "
                f"(intento {intento}/{MAX_REINTENTOS})..."
            )

            with requests.get(url, stream=True, timeout=60) as respuesta:
                respuesta.raise_for_status()

                #Escribir el archivo
                with open(ruta_destino, "wb") as archivo:
                    for bloque in respuesta.iter_content(chunk_size=CHUNK_SIZE):
                        if bloque:
                            archivo.write(bloque)

            print(f"Descarga completada: {ruta_destino}")
            return

        except Exception as e:
            print(f"Error: {e}")

            # Eliminar una descarga incompleta
            if os.path.exists(ruta_destino):
                os.remove(ruta_destino)

            if intento < MAX_REINTENTOS:
                time.sleep(5)
            else:
                raise

Descarga de los archivos

In [0]:
for url, nombre in zip(urls, archivos):

    destino = f"{RUTA_ZIP}/{nombre}"

    descargar_archivo(
        url=url,
        ruta_destino=destino
    )

Descargando AIS_2023_06_01.zip (intento 1/3)...
Descarga completada: /Volumes/workspace/oceanwatch/ais_data/zip/AIS_2023_06_01.zip
Descargando AIS_2023_06_02.zip (intento 1/3)...
Descarga completada: /Volumes/workspace/oceanwatch/ais_data/zip/AIS_2023_06_02.zip
Descargando AIS_2023_06_03.zip (intento 1/3)...
Error: ('Connection broken: IncompleteRead(41221782 bytes read, 271026243 more expected)', IncompleteRead(41221782 bytes read, 271026243 more expected))
Descargando AIS_2023_06_03.zip (intento 2/3)...
Descarga completada: /Volumes/workspace/oceanwatch/ais_data/zip/AIS_2023_06_03.zip
Descargando AIS_2023_06_04.zip (intento 1/3)...
Descarga completada: /Volumes/workspace/oceanwatch/ais_data/zip/AIS_2023_06_04.zip
Descargando AIS_2023_06_05.zip (intento 1/3)...
Descarga completada: /Volumes/workspace/oceanwatch/ais_data/zip/AIS_2023_06_05.zip
Descargando AIS_2023_06_06.zip (intento 1/3)...
Descarga completada: /Volumes/workspace/oceanwatch/ais_data/zip/AIS_2023_06_06.zip
Descargando A

La descarga de los siete archivos AIS se completó correctamente. Durante la descarga del archivo correspondiente al 3 de junio se presentó una interrupción, pero el proceso realizó un nuevo intento automáticamente y logró completar la descarga

Verificación de integridad

Antes de inicar con la exploración y el perfilamiento, se verifica si hay archivos dañados o incompletos en la descarga

In [0]:
import zipfile

def validar_zip(ruta_zip):

    try:
        with zipfile.ZipFile(ruta_zip, "r") as zip_file:

            # se recorren los archivos dentro del zip y comprueba si su CRC es correcto
            #
            archivo_corrupto = zip_file.testzip()

            if archivo_corrupto is not None:
                print(
                    f"ZIP corrupto: {os.path.basename(ruta_zip)} "
                    f"-> {archivo_corrupto}"
                )
                return False

        print(f"ZIP válido: {os.path.basename(ruta_zip)}")
        return True

    except zipfile.BadZipFile:
        print(f"ZIP inválido: {os.path.basename(ruta_zip)}")
        return False

Validación de los archivos

In [0]:
resultados_validacion = {}

for nombre in archivos:

    ruta = f"{RUTA_ZIP}/{nombre}"

    resultados_validacion[nombre] = validar_zip(ruta)

ZIP válido: AIS_2023_06_01.zip
ZIP válido: AIS_2023_06_02.zip
ZIP válido: AIS_2023_06_03.zip
ZIP válido: AIS_2023_06_04.zip
ZIP válido: AIS_2023_06_05.zip
ZIP válido: AIS_2023_06_06.zip
ZIP válido: AIS_2023_06_07.zip


Se encuentra que todos los archivos están correctos, se procede a realizar la descompresión de los archivos

## 2. Descompresión de los archivos

In [0]:
def descomprimir_zip(ruta_zip, ruta_destino):

    with zipfile.ZipFile(ruta_zip, "r") as zip_file:

        for archivo in zip_file.namelist():

            ruta_salida = os.path.join(ruta_destino, archivo)

            # Evitar extraer nuevamente archivos existentes
            if os.path.exists(ruta_salida):
                print(f"Ya existe: {archivo}")
                continue

            zip_file.extract(archivo, ruta_destino)
            print(f"Extraído: {archivo}")

Se almacenan los CSV validos en una carpeta independiente dentro del mismo VOLUME

In [0]:
for nombre, valido in resultados_validacion.items():

    if valido:
        ruta_zip = f"{RUTA_ZIP}/{nombre}"

        descomprimir_zip(
            ruta_zip=ruta_zip,
            ruta_destino=RUTA_CSV
        )

Extraído: AIS_2023_06_01.csv
Extraído: AIS_2023_06_02.csv
Extraído: AIS_2023_06_03.csv
Extraído: AIS_2023_06_04.csv
Extraído: AIS_2023_06_05.csv
Extraído: AIS_2023_06_06.csv
Extraído: AIS_2023_06_07.csv


Verifiación de la estructura de archivos

In [0]:
# Ver archivos ZIP descargados
print("ARCHIVOS ZIP:")
for archivo in dbutils.fs.ls(RUTA_ZIP):
    print(f"  - {archivo.name}")

print("\nARCHIVOS CSV DESCOMPRIMIDOS:")
for archivo in dbutils.fs.ls(RUTA_CSV):
    print(f"  - {archivo.name}")

ARCHIVOS ZIP:
  - AIS_2023_06_01.zip
  - AIS_2023_06_02.zip
  - AIS_2023_06_03.zip
  - AIS_2023_06_04.zip
  - AIS_2023_06_05.zip
  - AIS_2023_06_06.zip
  - AIS_2023_06_07.zip

ARCHIVOS CSV DESCOMPRIMIDOS:
  - AIS_2023_06_01.csv
  - AIS_2023_06_02.csv
  - AIS_2023_06_03.csv
  - AIS_2023_06_04.csv
  - AIS_2023_06_05.csv
  - AIS_2023_06_06.csv
  - AIS_2023_06_07.csv


Se realizó la descarga, validación y descompresión de los archivos de los siete días del análisis, Los archivos .zip y .csv se almacenaron en el volume del proyecto y quedan disponibles para lectura y procesamiento